# ColtraneToSheet — Colab runner

End-to-end wrapper around `run.py`. Clones the repo, installs system deps, lets you upload an audio file, runs the pipeline, and downloads the result.

**Before running:** Runtime → Change runtime type → **T4 GPU**. The first run downloads ~400 MB of model weights and takes a few minutes.

In [ ]:
# 1. Clone the repo (edit REPO_URL if you forked it). Idempotent: safe to re-run.
import os
REPO_URL = "https://github.com/paubernat/ColtraneToSheet.git"
REPO_DIR = "/content/ColtraneToSheet"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
# 2. System deps Colab doesn't ship: MuseScore (PDF render) + FluidSynth (MIDI playback).
# Python deps are auto-installed by run.py on first invocation.
!apt-get -qq update
!apt-get -qq install -y musescore3 fluidsynth > /dev/null

In [ ]:
# 3. Upload your audio file. Drops into input/.
import shutil
from pathlib import Path
from google.colab import files

Path("input").mkdir(exist_ok=True)
uploaded = files.upload()
INPUT_FILENAME = next(iter(uploaded))
shutil.move(INPUT_FILENAME, f"input/{INPUT_FILENAME}")
print(f"Saved to input/{INPUT_FILENAME}")

In [ ]:
# 4. Run the pipeline.
# SEP_MODEL options:
#   - "bs_roformer"  (default, BS-RoFormer 6-stem, higher quality piano, ~400 MB)
#   - "htdemucs_6s"  (Demucs 6-stem, faster, weaker on piano, ~55 MB)
SEP_MODEL = "bs_roformer"

!python run.py "input/{INPUT_FILENAME}" --sep-model {SEP_MODEL}

In [ ]:
# 5. Zip the output folder and download it.
from pathlib import Path
import shutil
from google.colab import files

input_stem = Path(INPUT_FILENAME).stem
run_dir = Path("output") / f"{input_stem}_{SEP_MODEL}"
print("Contents:")
for p in sorted(run_dir.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.0f} KB)")

zip_path = shutil.make_archive(str(run_dir), "zip", str(run_dir))
files.download(zip_path)